In [1]:
import pandas as pd
import numpy as np

In [2]:
from pytrends.request import TrendReq ###-----> Google Trend'ten veri çekmek 

### Google Trends sorgu için nesle oluşturuldu

In [3]:
pytrends = TrendReq(
    hl ="tr-TR",
    tz=0
)

In [4]:
keywords = ["LangChain"]

In [5]:
pytrends.build_payload(      ### build_payload ---> sorgu kapsamı
    kw_list=keywords,
    cat=0,
    timeframe="2023-08-11 2026-08-11",
    geo="",
    gprop=""
)

In [7]:
df = pytrends.interest_over_time()  ###---> Gerçek veri çekildi

In [ ]:
df

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.to_csv(
    "../data/raw/langchain_global_raw.csv"
)

In [ ]:
df.index.min()

In [ ]:
df.index.max()

In [ ]:
pd.infer_freq(df.index) ### ----> Verinin haftalık olduğu bulundu W --> Weekly / SUN --> Sunday

In [ ]:
df.index.to_series().diff().value_counts() ### Doğrulama işlemi 

In [ ]:
df.isnull().sum()  ### NaN kontrolü ama tarih kayıp ise yetersiz işlem

In [ ]:
df.index.duplicated().sum() ### Tarih tekrarı kontrolü 

In [ ]:
(df["LangChain"] == 0).sum() 

In [ ]:
df["LangChain"].describe()

In [ ]:
df.to_csv(
    "../data/raw/langchain_global_raw.csv"
)

In [ ]:
### teorik olarak olması gereken tarih takvimi oluşturuldu çünkü eksik tarih verisini daha basit gözlemyebilmek için
full_index = pd.date_range(    ### date_range --> düzenli tarih dizisi oluşturur
    start=df.index.min(),
    end=df.index.max()
)

full_index

In [ ]:
freq = pd.infer_freq(df.index) ### Zaman serisinin hangi frekansta ilerlediğini otomatik tespit ediyoruz

print("Tespit edilen frekans:", freq)

In [ ]:
### Veri setinin ilk ve son atrihi arasında buluması gereken tüm haftalık tarihleri oluşturuyoruz
### Amaç ise Gerçek veri içerisinde tamamen kayıp olan bir hafta varsa daha sonra buun tespit edebilmek

full_index = pd.date_range(
    start=df.index.min(),
    end=df.index.max(),
    freq=freq
)

In [ ]:
expected_count = len(full_index) ### teorik olarak bu zaman aralığında 158 hafta gözlem olması gerekiyor
print("Beklenen gözlem sayısı:", expected_count)

In [ ]:
actual_count = len(df) ### Google Trend'ten çektiğimiz gerçek gözlem sayısını kontrol ediyoruz ama hala yeterli dğeil
print("gerçek gözlem sayısı:", actual_count)

## Neden hala yeterli değil ? 

##### " 1 Ocak, 8 Ocak, 15 Ocak, 22 Ocak " olması gerekirken gerçek veri " 1 Ocak, 8 Ocak, 22 Ocak, 29 ocak " 

##### Her ikisinde de 4 tarih var ama 15 Ocak eksik, 29 Ocak ise fazladan





In [ ]:
### difference() -----> full index - gerçek index = eksik tarihler
missing_count = full_index.difference(df.index)  

In [ ]:
missing_count

In [ ]:
print("Eksik tarih sayısı:", len(missing_count))

## Interpolation ----> bilinen değerlerden bilinmeyeni tahmin etme yöntemi


#### Zaman serisinin haftalık frekansı kontrol edildi ve eksik timestamp tespit edilmedi. Bu nedenle interpolation uygulanmadı.

#### Ama eğer eksik değer herhangi bir hafta eksik olsaydı, o tarihte NaN değeri oluşacaktı. Bu yüzden zaman bazlı interpolation uyguluyoruz. Eksik değer yoksa veri üzerinde herhangi bir değişiklik yapmamış olacaz.




In [ ]:
## Veriyi, oluşturduğumuz düzenli haftalık tarih takvimine göre yeniden indeksliyoruz.
## Eğer gerçek veride tamamen eksik bir hafta varsa, o tarih DataFrame'e eklenir
## ve LangChain değeri NaN olarak görünür.

df_regular = df.reindex(full_index)
missing_count = df_regular["LangChain"].isna().sum()

print("Eksik gözlem sayısı:", missing_count)

In [ ]:
if missing_count > 0:
    df_regular["LangChain"] = (
        df_regular["LangChain"]
        .interpolate(method="time")
    )
    print("Eksik değerler interpolation ile dolduruldu.")

else:
     print("Eksik zaman noktası bulunmadı. Interpolation uygulanmadı.")
    

In [ ]:
clean_df = df.copy()

In [ ]:
## Google Trends'teki tamamlanmamış dönemlere bakılıyor..
## isPartial=True olan satır, ilgili haftanın henüz tamamlanmadığını gösterir.

clean_df[clean_df["isPartial"] == True]

In [ ]:
## Tamamlanmamış haftaları temiz veri setinden çıkarıyoruz.
## Çünkü henüz tamamlanmamış bir haftanın trend değeri ileride değişebilir
## ve modelleme aşamasında yanıltıcı olabilir.

clean_df = clean_df[
    clean_df["isPartial"] == False
].copy()

In [ ]:
clean_df.head()

In [ ]:
clean_df.tail()

In [ ]:
clean_df = clean_df.rename(
    columns={"LangChain": "trend_score"}
)

In [ ]:
clean_df.head()

In [ ]:
clean_df.index.name

In [ ]:
clean_df.to_csv(
    "../data/processed/langchain_global_clean.csv"
)

In [ ]:
import pyarrow as pa

pa.__version__

In [ ]:
clean_df.to_parquet(
    "../data/processed/langchain_global_clean.parquet"
)

# EDA

In [ ]:
### Model kurmadan önce zaman serisinin genel davranışını inceliyoruz.
### Buradaki amaç tren skorunun zaman içerisinde nasıl değiştiğini görmek

In [ ]:
import matplotlib.pyplot as plt

### TrendScore sütununu tarihe göre çiziyoruz.
### Index'imiz date olduğu için x ekseni otomatik olarak tarih oluyor.

clean_df["trend_score"].plot(figsize=(12,5))

plt.title("LangChain Google Trends - Global")
plt.xlabel("Date")
plt.ylabel("Trend Score")
plt.show()

In [ ]:
### Trend skorundaki kısa süreli dalgalamaları yumuşsatmak için 
### 4 haftalık hareketli ortalama(rolling mean) oluşturuyoruz
### window=4 ----> her satır için kendisi dahil son 4 haftasının ortalamasını hesaplar
### ilk üç değer Nan çıkabilir ama eksik veri problemi değil.


clean_df["rolling_4"] = (
    clean_df["trend_score"]
    .rolling(window=4)
    .mean()
)

In [ ]:
clean_df.head(10)

In [ ]:
### Gerçek tend skoru ile 4 haftalık hareketli ortalamayı
### aynı grafik üzerinde karşılaştırıyoruz.
##
### trend_score -> gerçek ve daha dalgalı seri
### rolling_4   -> kısa dönem dalgalanmaları yumuşatılmış seri

clean_df[
    ["trend_score", "rolling_4"]
].plot(figsize=(12, 5))

plt.title("LangChain Trend ve 4 Haftalık Hareketli Ortalama")
plt.xlabel("Date")
plt.ylabel("Trend Score")
plt.show()

In [ ]:
### LangChain'e olan ilginin en yüksek olduğu 10 hafta:

clean_df["trend_score"].nlargest(10)

In [ ]:
###LangChain'e olan ilginin en düşük olduğu 10 hafta:

clean_df["trend_score"].nsmallest(10)

## Decomposition ----> Zaman serisi verisini analiz ederek trend (genel eğilim) ve sezonsallık (seasonality - örn: hafta sonu düşüşleri) bileşenlerine ayırmak

#### ayırmak için statsmodels kütüphanesindeki seasonal_decompose fonksiyonunu kullanıyoruz.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

####  Additive	             /        Multiplicative
####  Etkiler toplanır	      /       Etkiler çarpılır
####  Dalgalanma yaklaşık sabit	  /   Dalgalanma seri büyüdükçe büyür
####  Trend + Seasonal + Residual / Trend × Seasonal × Residual

In [ ]:
decomposition = seasonal_decompose(
    clean_df["trend_score"],
    model="additive", ### Zaman serisini Trend + Sezonsallık + Artık (Residual) şeklinde parçala.
    
    period=52  ### ---> veri frekansımız haftalık (W-SUN) ve yaklaşık 52 hafta = 1 yıl
               ### ---> böylece yıllık olarak tekrar eden bir yapı olup olmadığını inceliyoruz
)

In [ ]:
## Decomposition sonucunda oluşan dört bileşeni görselleştiriyoruz:
##
## Observed -> gerçek seri
## Trend    -> uzun dönemli yön
## Seasonal -> tekrar eden dönemsel yapı
## Resid    -> modelin açıklayamadığı kalan hareket

decomposition.plot()
plt.show()